In [ ]:
import os
from typing import BinaryIO
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

from utils import read_range_thread, read_range_process

# %load_ext memory_profiler # если нужно замерить память в рамках ячейке (%%memit)

# Если необходимо сделать больший объем файла для чтения, чтобы проверить параллелизм на практике,
# То можно использовать следующую bash команду в папке ../tests/fixtures: for i in {1..400}; do cat tinystories_sample_5M.txt; done > tinystories_sample_2000M.txt

In [2]:
def find_chunk_boundaries(
    file: BinaryIO,
    desired_num_chunks: int,
    split_special_token: bytes,
) -> list[int]:
    """
    Chunk the file into parts that can be counted independently.
    May return fewer chunks if the boundaries end up overlapping.
    """
    assert isinstance(split_special_token, bytes), "Must represent special token as a bytestring"

    # Get total file size in bytes
    file.seek(0, os.SEEK_END)
    file_size = file.tell()
    file.seek(0)

    chunk_size = file_size // desired_num_chunks

    # Initial guesses for chunk boundary locations, uniformly spaced
    # Chunks start on previous index, don't include last index
    chunk_boundaries = [i * chunk_size for i in range(desired_num_chunks + 1)]
    chunk_boundaries[-1] = file_size

    mini_chunk_size = 4096  # Read ahead by 4k bytes at a time

    for bi in range(1, len(chunk_boundaries) - 1):
        initial_position = chunk_boundaries[bi]
        file.seek(initial_position)  # Start at boundary guess
        while True:
            mini_chunk = file.read(mini_chunk_size)  # Read a mini chunk

            # If EOF, this boundary should be at the end of the file
            if mini_chunk == b"":
                chunk_boundaries[bi] = file_size
                break

            # Find the special token in the mini chunk
            found_at = mini_chunk.find(split_special_token)
            if found_at != -1:
                chunk_boundaries[bi] = initial_position + found_at
                break
            initial_position += mini_chunk_size

    # Make sure all boundaries are unique, but might be fewer than desired_num_chunks
    return sorted(set(chunk_boundaries))

In [3]:
%%timeit
with open("../tests/fixtures/tinystories_sample_2000M.txt", "rb") as f:
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    results = []
    for start, end in zip(boundaries[:-1], boundaries[1:]):
        f.seek(start)
        chunk = f.read(end - start).decode("utf-8", errors="ignore")
        results.append(chunk)

690 ms ± 21.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
%%timeit
with open("../tests/fixtures/tinystories_sample_2000M.txt", "rb") as f:
    num_workers = 12
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    fd = f.fileno()

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(
            executor.map(
                read_range_thread,
                [fd] * (len(boundaries) - 1),
                boundaries[:-1],
                boundaries[1:],
            )
        )

498 ms ± 15.5 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
with open("../tests/fixtures/tinystories_sample_2000M.txt", "rb") as f:
    num_workers = 12
    boundaries = find_chunk_boundaries(f, 40, b"<|endoftext|>")

    fd = f.fileno()

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        results = list(
            executor.map(
                read_range_process,
                ["../tests/fixtures/tinystories_sample_2000M.txt"] * (len(boundaries) - 1),
                boundaries[:-1],
                boundaries[1:],
            )
        )

488 ms ± 8.22 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
